In [ ]:
%pip install psycopg2-binary --quiet

In [ ]:
import boto3, json

dbutils.widgets.text("service_credential", "rema_prd_service_credential")
SERVICE_CREDENTIAL_NAME = dbutils.widgets.get("service_credential").strip() or "rema_prd_service_credential"
REGION = "eu-west-1"

provider = dbutils.credentials.getServiceCredentialsProvider(SERVICE_CREDENTIAL_NAME)

session = boto3.Session(
    botocore_session=provider,
    region_name=REGION
)

sm = session.client("secretsmanager")
response = sm.get_secret_value(SecretId="rds/case/prod")

# Parse the secret - adjust keys based on actual secret structure
secret = json.loads(response["SecretString"]) if isinstance(response.get("SecretString"), str) else {"password": response["SecretString"]}
print("Secret keys available:", list(secret.keys()))

In [ ]:
import psycopg2
import pandas as pd
from datetime import datetime, timezone

# ============================================================
# CONFIGURATION
# ============================================================
# Job parameters, with the dev values as defaults so the notebook still runs
# interactively. The job passes these so one notebook serves every target.
dbutils.widgets.text("target_catalog", "turbine_lens_dev")
dbutils.widgets.text("target_schema", "rema_prod")
TARGET_CATALOG = dbutils.widgets.get("target_catalog").strip() or "turbine_lens_dev"
TARGET_SCHEMA = dbutils.widgets.get("target_schema").strip() or "rema_prod"

# RETIRED UPSTREAM, 2026-09-15. These four were dropped from the REMA source around
# early July 2026 and are deliberately NOT listed below:
#   global_business_unit, organisation_unit,
#   user_global_business_unit, user_organisation_unit
# Postgres answers `relation "public.<name>" does not exist` for each, which is the
# absent-table error and not the permission-denied one. Nothing downstream reads them:
# no Genie space, no view DDL, no server code (checked). The one apparent hit,
# `global_business_unit_code`, is a COLUMN in the REX space and unrelated.
#
# HOW THIS WENT UNNOTICED FOR TWO MONTHS, which is the part worth remembering: the
# per-table loop below swallowed all four failures into `failures` and the cell still
# ended green, so the noprod nightly job reported success every night while these four
# Delta tables sat frozen at their June/July contents — stale data presented as current.
# The FAIL LOUDLY guard at the end of this cell is what surfaced it. Re-adding a name
# here without checking the source will reintroduce a hard failure, now loudly.
# Primary keys per table (all 27 tables, excluding alembic_version)
TABLE_PKS = {
    "action": ["id"],
    "action_priority": ["id"],
    "action_status": ["id"],
    "assessment": ["id"],
    "component": ["id"],
    "country": ["id"],
    "file": ["id"],
    "initiative": ["id"],
    "initiative_component": ["initiative_id", "component_id"],
    "initiative_country": ["initiative_id", "country_id"],
    "initiative_plant": ["initiative_id", "plant_id"],
    "initiative_region": ["initiative_id", "region_id"],
    "initiative_status": ["id"],
    "link": ["id"],
    "link_type": ["id"],
    "plant": ["id"],
    "region": ["id"],
    "role": ["id"],
    "service_provider": ["id"],
    "technology": ["id"],
    "user": ["id"],
    "user_action": ["user_id", "action_id"],
    "user_initiative": ["user_id", "initiative_id"],
}

# ============================================================
# CONNECTION
# ============================================================
def connect_pg():
    return psycopg2.connect(
        host="example.invalid",
        port=secret.get("DB_PORT", 5432),
        dbname=secret.get("DB_NAME", secret.get("dbname", "postgres")),
        user=secret.get("DB_USER", secret.get("username", "")),
        password=secret.get("DB_PASSWORD", secret.get("password", ""))
    )

# ============================================================
# SYNC: Full Refresh + MERGE (one strategy for all tables)
# ============================================================
def sync_table(conn, table_name, pk_cols):
    """Read entire source table, MERGE into Delta target.
    Handles inserts, updates, and deletes in one pass.
    Returns a dict with operation metrics."""
    target = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{table_name}"

    # Full read from PostgreSQL
    pdf = pd.read_sql(f'SELECT * FROM public."{table_name}"', conn)

    if pdf.empty:
        print(f"  SKIP  {table_name} (0 rows in source)")
        return {"table": table_name, "inserted": 0, "updated": 0, "deleted": 0, "status": "skipped"}

    # First run: create table if it doesn't exist yet
    if not spark.catalog.tableExists(target):
        sdf = spark.createDataFrame(pdf)
        sdf.write.format("delta").mode("overwrite").saveAsTable(target)
        print(f"  CREATE {target} ({len(pdf)} rows) [initial load]")
        return {"table": table_name, "inserted": len(pdf), "updated": 0, "deleted": 0, "status": "created"}

    # Subsequent runs: MERGE (upsert + delete)
    sdf = spark.createDataFrame(pdf)
    sdf.createOrReplaceTempView(f"_src_{table_name}")

    merge_on = " AND ".join([f"target.`{pk}` = source.`{pk}`" for pk in pk_cols])
    non_pk_cols = [c for c in pdf.columns if c not in pk_cols]

    # Build MERGE statement.
    #
    # `UPDATE SET *` / `INSERT *` rather than an explicit column list, and that is what
    # makes SCHEMA EVOLUTION actually work. Evolution only adds a column when the clause
    # is the STAR form: naming `target.OstID` explicitly makes Spark resolve it against
    # the target BEFORE evolution can create it, so the statement dies at analysis with
    #   [DELTA_MERGE_UNRESOLVED_EXPRESSION] Cannot resolve target.OstID in UPDATE clause
    # even with `WITH SCHEMA EVOLUTION` present — measured 2026-09-15, where adding the
    # clause alone changed the error text from `given columns target.…` to
    # `given columns source.…` and otherwise failed identically. Two changes, one fix.
    #
    # The star form is safe here because source and target carry the same column names by
    # construction: the source view is a straight `SELECT *` of the Postgres table, so the
    # explicit lists this replaces were only ever spelling out "all columns".
    if non_pk_cols:
        merge_sql = f"""
            MERGE WITH SCHEMA EVOLUTION INTO {target} AS target
            USING _src_{table_name} AS source
            ON {merge_on}
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
            WHEN NOT MATCHED BY SOURCE THEN DELETE
        """
    else:
        # Junction tables with only PK columns: nothing to update, so no UPDATE clause
        # (an `UPDATE SET *` here would just set the keys to themselves).
        merge_sql = f"""
            MERGE WITH SCHEMA EVOLUTION INTO {target} AS target
            USING _src_{table_name} AS source
            ON {merge_on}
            WHEN NOT MATCHED THEN INSERT *
            WHEN NOT MATCHED BY SOURCE THEN DELETE
        """

    # Execute MERGE and capture metrics
    result = spark.sql(merge_sql)
    metrics = result.collect()[0]
    
    inserted = metrics["num_inserted_rows"]
    updated = metrics["num_updated_rows"]
    deleted = metrics["num_deleted_rows"]

    # Print with details
    changes = []
    if inserted > 0:
        changes.append(f"+{inserted} inserted")
    if updated > 0:
        changes.append(f"~{updated} updated")
    if deleted > 0:
        changes.append(f"-{deleted} deleted")
    
    if changes:
        print(f"  MERGE {table_name:<30} {' | '.join(changes)}")
    else:
        print(f"  MERGE {table_name:<30} (no changes)")

    return {"table": table_name, "inserted": inserted, "updated": updated, "deleted": deleted, "status": "synced"}

# ============================================================
# RUN
# ============================================================
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_CATALOG}.{TARGET_SCHEMA}")

# Make the schema readable by the Turbine Lens group, in the same breath as creating it.
# The app's own service principal is a member, and it is the app that queries these
# tables through the REMA/REX Genie spaces at answer time — so without this the space
# answers nothing and does not error.
#
# It belongs HERE rather than in the bundle because the bundle does not own this schema:
# the notebook creates it, so the ingest SP owns it, and only the owner can grant. That
# is also the trap this line closes. Measured 2026-09-15: both schemas had been created
# INTERACTIVELY by a human, so `CREATE SCHEMA IF NOT EXISTS` was a no-op and the job SP
# could not write into a schema it did not own — 27 of 27 and 43 of 43 tables failed on
# `USE SCHEMA`. Ownership follows whoever creates first, so let it always be this job.
#
# GRANT is additive, unlike the bundle's declarative `grants:` which revokes anything
# undeclared, so re-running this cannot take a privilege away from anyone.
# Tolerated on purpose, and the ONLY failure in this notebook that is. Only a schema's
# owner may GRANT, and ownership follows whoever created it first: in prod this job
# creates the schema and owns it, so the grant lands; in noprod both schemas were created
# BY HAND long ago, so the job SP holds no MANAGE and this raises
# `PERMISSION_DENIED: User does not have MANAGE on Schema ...` — measured 2026-09-15,
# where it failed the whole nightly sync until this guard was added.
#
# Why tolerating THIS differs from tolerating a table failure: a missing grant cannot
# corrupt or stale the data, and it surfaces immediately and loudly the first time the app
# queries the space. A swallowed table failure does the opposite — it leaves a stale table
# looking current, which is how four REMA tables went two months unnoticed.
#
# The real fix is to converge ownership so both environments behave alike; until then this
# keeps a privilege nicety from taking down the data movement it is attached to.
try:
    spark.sql(
        f"GRANT USE SCHEMA, SELECT ON SCHEMA {TARGET_CATALOG}.{TARGET_SCHEMA} "
        f"TO `portfolio-org0004-default-turbine_lens`"
    )
    print(f"  GRANT USE SCHEMA, SELECT on {TARGET_CATALOG}.{TARGET_SCHEMA} -> portfolio-org0004-default-turbine_lens")
except Exception as _grant_err:
    print(f"  WARN  could not grant on {TARGET_CATALOG}.{TARGET_SCHEMA}: {_grant_err}")
    print("  WARN  this job does not own that schema; the app's read access must be granted "
          "by its owner. The sync itself is unaffected.")

run_start = datetime.now(timezone.utc)
print(f"{'='*70}")
print(f"FULL REFRESH + MERGE — {run_start.strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"{'='*70}\n")

# Let MERGE absorb columns the source has gained.
# SCHEMA EVOLUTION is requested per-statement, in the MERGE below, via
# `MERGE WITH SCHEMA EVOLUTION INTO`. The obvious alternative does NOT work here:
# `spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")` is rejected
# outright on serverless compute —
#   [CONFIG_NOT_AVAILABLE.SERVERLESS_DELTA_SCHEMA_AUTO_MERGE_ENABLED] ... not supported in
#   serverless environments. Use the write option instead: .option("mergeSchema", "true")
# measured 2026-09-15 — and it fails at the CONF CALL, before any table is touched, so it
# takes down tasks that had nothing to do with schema drift. The suggested write option is
# a DataFrame-write option and does not apply to a SQL MERGE, which is what this notebook
# issues. The SQL clause is the serverless-safe equivalent.
# The MERGE below builds
# `UPDATE SET target.<col> = source.<col>` from the SOURCE columns, so a column added
# upstream makes the statement unresolvable against an existing target:
#   [DELTA_MERGE_UNRESOLVED_EXPRESSION] Cannot resolve target.OstID in UPDATE clause
# Measured 2026-09-15 on noprod `rex_prod.Rex` (36 columns) after `OstID` appeared in the
# source (37). The same day PROD synced the same table fine — not because prod was
# healthier, but because its table had just been created from scratch and therefore
# matched the source. Prod carried the identical latent bug, due on the next added column.
#
# What this does NOT do, deliberately: a column REMOVED upstream leaves a now-stale column
# on the target rather than dropping it, and an incompatible TYPE change still fails. Both
# are the right side of the trade — the first is visible and harmless, the second is a real
# incompatibility that should stop the run rather than be papered over.
conn = connect_pg()
all_metrics = []
failures = []

for table_name, pk_cols in TABLE_PKS.items():
    try:
        metrics = sync_table(conn, table_name, pk_cols)
        all_metrics.append(metrics)
    except Exception as e:
        print(f"  FAIL  {table_name}: {e}")
        failures.append((table_name, str(e)))

conn.close()

# ============================================================
# SUMMARY REPORT
# ============================================================
run_end = datetime.now(timezone.utc)
total_inserted = sum(m["inserted"] for m in all_metrics)
total_updated = sum(m["updated"] for m in all_metrics)
total_deleted = sum(m["deleted"] for m in all_metrics)
tables_with_changes = [m for m in all_metrics if m["inserted"] + m["updated"] + m["deleted"] > 0]

print(f"\n{'='*70}")
print(f"SYNC SUMMARY")
print(f"{'='*70}")
print(f"  Duration:     {(run_end - run_start).total_seconds():.1f}s")
print(f"  Tables:       {len(TABLE_PKS)} total, {len(tables_with_changes)} with changes")
print(f"  Inserted:     {total_inserted} rows")
print(f"  Updated:      {total_updated} rows")
print(f"  Deleted:      {total_deleted} rows")
if failures:
    print(f"  Failures:     {len(failures)}")
    for name, err in failures:
        print(f"    - {name}: {err}")
print(f"{'='*70}")

# Display as a summary table
if tables_with_changes:
    print(f"\n\nTables with changes:")
    changes_df = pd.DataFrame(tables_with_changes)
    changes_df = changes_df[changes_df['status'] != 'skipped']
    changes_df = changes_df.sort_values(by=['inserted', 'updated', 'deleted'], ascending=False)
    display(changes_df)

# ============================================================
# FAIL LOUDLY
# ============================================================
# The per-table loop above catches EVERY exception into `failures`, so without
# this the cell ends green no matter how much failed. Measured in prod
# 2026-09-15: all 27 tables failed, the task was reported as a successful sync
# of zero tables, and the only signal was the NEXT cell crashing on a table
# that had never been created. A scheduled run has no such witness, and this
# job carries no failure mail.
#
# Raises on ANY failure, not only a total one: every run is a full read plus
# MERGE so a retry converges, and the task already retries once — a table that
# fails twice is a real problem. A partially-synced schema feeding a Genie
# space is worse than a red job, because the space answers confidently from
# whatever did land.
#
# The reason goes in the exception MESSAGE on purpose: `databricks jobs
# get-run-output` returns error/error_trace but NOT the prints above, and the
# run export carries no cell output either — so a printed reason is invisible
# to everything except a human on the run page.
if failures:
    _detail = "; ".join(f"{_n}: {_e[:300]}" for _n, _e in failures[:5])
    _more = f" (+{len(failures) - 5} more)" if len(failures) > 5 else ""
    raise RuntimeError(
        f"{len(failures)} of {len(TABLE_PKS)} tables failed to sync into "
        f"{TARGET_CATALOG}.{TARGET_SCHEMA}; {len(all_metrics)} succeeded. "
        f"{_detail}{_more}"
    )


In [ ]:
# ============================================================
# POST-SYNC CLEANUP — runs as part of the job
# ============================================================
# Strips HTML/Quill markup the source application stores in the description columns.
#
# This WAS a `%sql` cell, which meant it ran on every scheduled run against a
# HARDCODED `turbine_lens_dev.rema_prod.initiative` — so the moment this job ran in
# another workspace it would have UPDATEd noprod's table. A `%sql` cell cannot see
# Python variables, so parameterizing it required moving it to spark.sql().
#
# The statements are RAW f-strings (rf"""). The SQL contains \n, \t and \s, and a
# non-raw string would have Python resolve those escapes before SQL ever saw them —
# turning `'[ \t]+'` from a two-character regex class into a literal tab.
_STATEMENTS = [
rf"""
UPDATE {TARGET_CATALOG}.{TARGET_SCHEMA}.initiative
SET description_issue = 
  TRIM(BOTH '\n' FROM
    regexp_replace(
      regexp_replace(
        regexp_replace(
          regexp_replace(
            regexp_replace(
              regexp_replace(
                regexp_replace(
                  regexp_replace(
                    regexp_replace(
                      regexp_replace(description_issue,
                        '<img[^>]*>', ''),            -- 1. Remove embedded images (base64)
                      '<br\\s*/?>', '\n'),             -- 2. <br> to newline
                    '</p>|</h[1-6]>|</li>', '\n'),    -- 3. Block-closing tags to newline
                  '</ol>|</ul>', '\n'),               -- 4. List-closing tags to newline
                '<[^>]+>', ''),                        -- 5. Strip all remaining HTML tags
              '&nbsp;', ' '),                          -- 6. Decode &nbsp;
            '&amp;', '&'),                             -- 7. Decode &amp;
          '&lt;', '<'),                                -- 8. Decode &lt;
        '&gt;', '>'),                                  -- 9. Decode &gt;
      '[ \\t]+', ' ')                                  -- 10. Collapse multiple spaces/tabs
  )
WHERE description_issue IS NOT NULL
  AND description_issue LIKE '%<%';

-- Clean description_conclusion
""",
rf"""
UPDATE {TARGET_CATALOG}.{TARGET_SCHEMA}.initiative
SET description_conclusion = 
  TRIM(BOTH '\n' FROM
    regexp_replace(
      regexp_replace(
        regexp_replace(
          regexp_replace(
            regexp_replace(
              regexp_replace(
                regexp_replace(
                  regexp_replace(
                    regexp_replace(
                      regexp_replace(description_conclusion,
                        '<img[^>]*>', ''),            -- 1. Remove embedded images (base64)
                      '<br\\s*/?>', '\n'),             -- 2. <br> to newline
                    '</p>|</h[1-6]>|</li>', '\n'),    -- 3. Block-closing tags to newline
                  '</ol>|</ul>', '\n'),               -- 4. List-closing tags to newline
                '<[^>]+>', ''),                        -- 5. Strip all remaining HTML tags
              '&nbsp;', ' '),                          -- 6. Decode &nbsp;
            '&amp;', '&'),                             -- 7. Decode &amp;
          '&lt;', '<'),                                -- 8. Decode &lt;
        '&gt;', '>'),                                  -- 9. Decode &gt;
      '[ \\t]+', ' ')                                  -- 10. Collapse multiple spaces/tabs
  )
WHERE description_conclusion IS NOT NULL
  AND description_conclusion LIKE '%<%'
""",
]

for _i, _sql in enumerate(_STATEMENTS, 1):
    spark.sql(_sql)
    print(f"  [{_i}/{len(_STATEMENTS)}] applied")


In [ ]:
# ============================================================
# VIEW DEFINITIONS — now created by the job
# ============================================================
# These are what the `rema_qualitative_context` Genie space reads; it never queries the
# base tables. They previously sat inside a `/* ... */` block marked "for traceability
# only - DO NOT RUN", so they were deployed and inert: a fresh environment got 27 tables
# and a Genie space pointing at views that did not exist.
#
# Verified before enabling: each definition here is byte-identical to the live view in
# noprod (case-sensitive comparison against information_schema.views), so turning them on
# recreates exactly what is already there rather than redefining it.
#
# CREATE OR REPLACE is idempotent and a view stores no data, so running this every sync
# costs nothing and makes the views self-healing.
_STATEMENTS = [
rf"""
CREATE OR REPLACE VIEW {TARGET_CATALOG}.{TARGET_SCHEMA}.v_initiative_details AS
SELECT DISTINCT
  i.id AS initiative_id,
  i.title,
  i.date AS event_date,
  i.description_issue,
  i.description_conclusion,
  i.gain_lose_mwh,
  i.total_gain_lose_mwh,
  i.contractual_company_name,
  i.created_date,
  i.updated_date,
  ist.name AS status,
  a.name AS assessment,
  sp.name AS service_provider,
  CONCAT(u.firstname, ' ', u.lastname) AS created_by,
  u.email AS created_by_email,
  p.name AS plant_name,
  p.capacity_mw AS plant_capacity_mw,
  t.name AS technology,
  c.name AS country,
  r.name AS region,
  comp.name AS component
FROM {TARGET_CATALOG}.{TARGET_SCHEMA}.initiative i
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.initiative_status ist ON i.initiative_status_id = ist.id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.assessment a ON i.assessment_id = a.id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.service_provider sp ON i.service_provider_id = sp.id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.`user` u ON i.created_by_user_id = u.id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.initiative_plant ip ON i.id = ip.initiative_id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.plant p ON ip.plant_id = p.id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.technology t ON p.technology_id = t.id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.initiative_country ic ON i.id = ic.initiative_id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.country c ON ic.country_id = c.id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.initiative_region ir ON i.id = ir.initiative_id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.region r ON ir.region_id = r.id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.initiative_component icomp ON i.id = icomp.initiative_id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.component comp ON icomp.component_id = comp.id;

-- v_action_details: aggregated (one row per action, assigned users as list)
""",
rf"""
CREATE OR REPLACE VIEW {TARGET_CATALOG}.{TARGET_SCHEMA}.v_action_details AS
SELECT
  act.id AS action_id,
  act.title AS action_title,
  act.description AS action_description,
  act.deadline,
  act.closure_date,
  act.created_date,
  act.updated_date,
  ast.name AS action_status,
  ap.name AS action_priority,
  ap.variant AS priority_variant,
  act.initiative_id,
  i.title AS initiative_title,
  ist.name AS initiative_status,
  CONCAT(creator.firstname, ' ', creator.lastname) AS created_by,
  creator.email AS created_by_email,
  CONCAT_WS(', ', COLLECT_SET(CONCAT(assigned.firstname, ' ', assigned.lastname))) AS assigned_users,
  CONCAT_WS(', ', COLLECT_SET(assigned.email)) AS assigned_user_emails,
  COUNT(DISTINCT assigned.id) AS assigned_user_count
FROM {TARGET_CATALOG}.{TARGET_SCHEMA}.action act
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.action_status ast ON act.action_status_id = ast.id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.action_priority ap ON act.action_priority_id = ap.id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.initiative i ON act.initiative_id = i.id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.initiative_status ist ON i.initiative_status_id = ist.id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.`user` creator ON act.created_by_user_id = creator.id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.user_action ua ON act.id = ua.action_id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.`user` assigned ON ua.user_id = assigned.id
GROUP BY
  act.id, act.title, act.description, act.deadline, act.closure_date,
  act.created_date, act.updated_date,
  ast.name, ap.name, ap.variant,
  act.initiative_id, i.title, ist.name,
  creator.firstname, creator.lastname, creator.email
""",
]

for _i, _sql in enumerate(_STATEMENTS, 1):
    spark.sql(_sql)
    print(f"  [{_i}/{len(_STATEMENTS)}] applied")
